# NanoIndex Quick Start

Turn any PDF into a searchable tree. Ask questions, get answers with page citations.

No vector databases. No chunk tuning. No embeddings.

**What you need:**
- A Nanonets API key (free, 10K pages) — [get one here](https://docstrange.nanonets.com/app)
- An LLM API key (OpenAI, Anthropic, or Google) — for answering questions
- A PDF you want to query

## Step 1: Install

In [ ]:
!pip install nanoindex -q

In [ ]:
import os

# Paste your keys here
os.environ["NANONETS_API_KEY"] = ""   # Get from https://docstrange.nanonets.com/app
os.environ["ANTHROPIC_API_KEY"] = ""     # Or use OPENAI_API_KEY / GOOGLE_API_KEY

assert os.environ["NANONETS_API_KEY"], "Please set your Nanonets API key above"
assert os.environ["ANTHROPIC_API_KEY"], "Please set your LLM API key above"
print("Keys set.")

## Step 3: Upload a PDF

Upload any PDF you want to query. Or use the sample below.

In [ ]:
# Option A: Upload your own PDF
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_path}")

In [ ]:
from nanoindex import NanoIndex

# Auto-detects API keys from env vars
ni = NanoIndex()

# Or pick your LLM explicitly:
# ni = NanoIndex(llm="anthropic:claude-sonnet-4-6")
# ni = NanoIndex(llm="openai:gpt-5.4")
# ni = NanoIndex(llm="groq:llama-3.3-70b-versatile")

tree = ni.index(pdf_path)
print(f"Indexed: {tree.doc_name}")


## Step 4: Index the document

This sends the PDF to Nanonets OCR-3, which extracts the structure (headings, sections, tables). Then NanoIndex builds a tree from that structure. No LLM calls needed for this step.

## Step 5: Explore the tree

The tree shows how NanoIndex understood your document's structure.

In [ ]:
from nanoindex.utils.tree_ops import iter_nodes

nodes = list(iter_nodes(tree.structure))
print(f"Tree: {len(nodes)} nodes\n")

for node in nodes:
    indent = "  " * node.level
    bboxes = len(node.bounding_boxes)
    print(f"{indent}[{node.node_id}] {node.title}")
    print(f"{indent}  Pages {node.start_index}-{node.end_index} | {bboxes} bounding boxes")
    if node.summary:
        print(f"{indent}  Summary: {node.summary[:80]}...")
    print()

## Step 6: Ask questions

Now ask questions about the document. The LLM navigates the tree to find relevant sections and generates an answer with page citations.

In [ ]:
questions = [
    "What was the revenue?",
    "Which segment grew the fastest?",
    "What is the full year guidance?",
]

for q in questions:
    answer = ni.ask(q, tree)
    pages = sorted({p for c in answer.citations for p in c.pages})
    
    print(f"Q: {q}")
    print(f"A: {answer.content[:200]}")
    print(f"Pages cited: {pages}")
    print(f"Sections: {[c.title for c in answer.citations]}")
    print("-" * 60)

## Step 7: See the citations

Each answer comes with citations that tell you exactly where in the document the information came from. If bounding boxes are available, you get pixel-level coordinates.

In [ ]:
answer = ni.ask("What was the revenue and gross margin?", tree)

print("Answer:")
print(answer.content)
print(f"\nCitations ({len(answer.citations)}):")
for c in answer.citations:
    print(f"  Section: {c.title}")
    print(f"  Pages: {c.pages}")
    print(f"  Node ID: {c.node_id}")
    if c.bounding_boxes:
        print(f"  Bounding boxes ({len(c.bounding_boxes)}):")
        for bb in c.bounding_boxes[:3]:
            print(f"    Page {bb.page}: ({bb.x:.2f}, {bb.y:.2f}) {bb.width:.2f}x{bb.height:.2f}")
            if bb.text:
                print(f"    Text: {bb.text[:60]}")
    print()

## Step 8: Highlight citations on the PDF

Render the cited page with bounding boxes highlighted.

## Step 9: Explore the Entity Graph

Every document gets a knowledge graph built automatically using spaCy NLP (free, local). If a reasoning LLM is available, the graph is enhanced with LLM-extracted entities.

The graph captures entities (companies, people, money, dates) and relationships from the document.

In [ ]:
# The graph was built during indexing — let's explore it
graph = ni._graphs.get(tree.doc_name)

if graph:
    print(f"Entities: {len(graph.entities)}")
    print(f"Relationships: {len(graph.relationships)}")
    print()
    
    # Show entities grouped by type
    from collections import Counter
    types = Counter(e.entity_type for e in graph.entities)
    print("Entity types:")
    for t, count in types.most_common():
        print(f"  {t}: {count}")
    print()
    
    # Show top entities (most connected)
    print("Top entities:")
    for e in sorted(graph.entities, key=lambda x: len(x.source_node_ids), reverse=True)[:8]:
        print(f"  [{e.entity_type}] {e.name} (found in {len(e.source_node_ids)} sections)")
        if e.description:
            print(f"    {e.description[:80]}")
    print()
    
    # Show relationships
    print("Relationships:")
    for r in graph.relationships[:8]:
        print(f"  {r.source} --{r.keywords}--> {r.target}")
else:
    print("No graph available")


## Step 10: Fast Mode (graph-based retrieval)

The default mode (`agentic_vision`) gives highest accuracy but uses 6+ LLM calls. Fast mode uses the entity graph to find relevant sections with just 2 LLM calls — 3x cheaper.

Fast mode works by matching entities from your question against the graph, expanding via relationships, then having the LLM pick from a small set of candidates.

In [ ]:
# Compare: agentic (default) vs fast mode
import time

# Default: agentic_vision (highest accuracy, more LLM calls)
t0 = time.time()
answer_agentic = ni.ask("What was the revenue?", tree, pdf_path=pdf_path)
agentic_time = time.time() - t0

# Fast: graph-based retrieval (cheaper, fewer LLM calls)
t0 = time.time()
answer_fast = ni.ask("What was the revenue?", tree, mode="fast")
fast_time = time.time() - t0

print(f"Agentic ({agentic_time:.1f}s): {answer_agentic.content[:100]}")
print(f"Fast    ({fast_time:.1f}s): {answer_fast.content[:100]}")
print(f"\nSpeedup: {agentic_time/max(fast_time, 0.1):.1f}x faster with fast mode")


## Step 11: Save and reuse

Save the tree so you don't have to re-extract every time.

In [ ]:
from nanoindex.utils.tree_ops import save_tree, load_tree

# Save
save_tree(tree, "my_tree.json")
print("Saved tree to my_tree.json")

# Load (instant, no API call)
tree2 = load_tree("my_tree.json")
print(f"Loaded: {tree2.doc_name} with {len(list(iter_nodes(tree2.structure)))} nodes")

# You can keep asking questions on the loaded tree
answer = ni.ask("Summarize this document in 2 sentences", tree2)
print(f"\n{answer.content}")

## Step 12: Build a Knowledge Base (optional)

Add multiple documents to a knowledge base. Browse it like a wiki in Obsidian.

In [ ]:
from nanoindex import KnowledgeBase

kb = KnowledgeBase(
    "./my_kb",
    llm="anthropic:claude-sonnet-4-6",
)

# Add your document
kb.add(pdf_path)

# Check status
status = kb.status()
print(f"Documents: {status['documents']}")
print(f"Concepts: {status['concepts']}")

# See the generated wiki files
import os
for root, dirs, files_list in os.walk("./my_kb"):
    dirs[:] = [d for d in dirs if d != ".nanoindex"]
    for f in sorted(files_list):
        print(f"  {os.path.relpath(os.path.join(root, f), './my_kb')}")

---

## What's next?

- **CLI**: `nanoindex ask report.pdf "What was revenue?" --llm-model gpt-5.4`
- **Dashboard**: `nanoindex viz tree.json` opens an interactive tree + graph viewer
- **Multiple docs**: Add more PDFs to the Knowledge Base and query across all of them
- **GitHub**: [github.com/nanonets/nanoindex](https://github.com/nanonets/nanoindex)
- **Docs**: [nanonets.com/research/nanonets-ocr-3](https://nanonets.com/research/nanonets-ocr-3)